# Tau `category2=1` Events

Load the Tau category CSV and list the events that ended up in `second_category/category1`.

In [1]:
from pathlib import Path

import pandas as pd

CSV_PATH = Path(
    "/project/def-nahee/kbas/Graphnet-Applications/Metadata/CategoryInformation/"
    "String340MC/Full_Geometry_Tau_category2_events.csv"
)

df = pd.read_csv(CSV_PATH)

category1_events = (
    df.loc[df["category_value"].astype(str) == "1", ["RunID", "EventID"]]
    .sort_values(["RunID", "EventID"])
    .reset_index(drop=True)
)

pd.set_option("display.max_rows", None)
print(f"n_events = {len(category1_events)}")
display(category1_events)


n_events = 52


,RunID,EventID
0,4,13
1,4,15
2,4,34
3,4,50
4,4,52
5,4,54
6,4,61
7,4,65
8,4,70
9,4,73


In [2]:
unique_run_ids = sorted(category1_events["RunID"].unique().tolist())

print(f"n_unique_RunID = {len(unique_run_ids)}")
print(unique_run_ids)


n_unique_RunID = 2
[4, 5]


In [3]:
run_ids = [4,5]

tau_raw_base = Path(
    "/project/6008051/pone_simulation/"
    "MC000004-nu_tau-2_7-LeptonInjector_PROPOSAL_clsim-v10/Generator"
)

tau_file_paths = [tau_raw_base / f"gen_{run_id:03d}.i3.zst" for run_id in run_ids]

for path in tau_file_paths:
    print(path)


/project/6008051/pone_simulation/MC000004-nu_tau-2_7-LeptonInjector_PROPOSAL_clsim-v10/Generator/gen_004.i3.zst
/project/6008051/pone_simulation/MC000004-nu_tau-2_7-LeptonInjector_PROPOSAL_clsim-v10/Generator/gen_005.i3.zst


In [4]:
from icecube import dataio, LeptonInjector

event_properties_fields = [
    "totalEnergy",
    "zenith",
    "azimuth",
    "finalStateX",
    "finalStateY",
    "finalType1",
    "finalType2",
    "initialType",
    "x",
    "y",
    "z",
    "totalColumnDepth",
    "radius",
    "impactParameter",
]

rows = []

for tau_file_path in tau_file_paths:
    print(f"reading {tau_file_path}")
    i3_file = dataio.I3File(str(tau_file_path))
    n_daq = 0

    while i3_file.more():
        try:
            frame = i3_file.pop_daq()
        except Exception as exc:
            if "no frame to pop" in str(exc):
                break
            raise

        n_daq += 1

        if "I3EventHeader" not in frame or "EventProperties" not in frame:
            continue

        header = frame["I3EventHeader"]
        ep = frame["EventProperties"]

        row = {
            "raw_file": tau_file_path,
            "daq_index": n_daq,
            "RunID": header.run_id,
            "SubrunID": header.sub_run_id,
            "EventID": header.event_id,
            "SubEventID": header.sub_event_id,
        }
        for field in event_properties_fields:
            row[field] = getattr(ep, field)

        rows.append(row)

raw_event_properties_df = (
    pd.DataFrame(rows)
    .sort_values(["RunID", "EventID", "SubEventID"])
    .reset_index(drop=True)
)

pd.set_option("display.max_rows", None)
print(f"n_rows = {len(raw_event_properties_df)}")
display(raw_event_properties_df)


reading /project/6008051/pone_simulation/MC000004-nu_tau-2_7-LeptonInjector_PROPOSAL_clsim-v10/Generator/gen_004.i3.zst
reading /project/6008051/pone_simulation/MC000004-nu_tau-2_7-LeptonInjector_PROPOSAL_clsim-v10/Generator/gen_005.i3.zst
n_rows = 400


,raw_file,daq_index,RunID,SubrunID,EventID,SubEventID,totalEnergy,zenith,azimuth,finalStateX,finalStateY,finalType1,finalType2,initialType,x,y,z,totalColumnDepth,radius,impactParameter
0,/project/6008051/pone_simulation/MC000004-nu_t...,1,4,4294967295,1,0,278404.465310,0.762805,1.795441,0.003889,0.335883,-12,-2000001006,-12,-582.135794,953.639166,1611.977544,3.772313e+05,1117.277826,497.168506
1,/project/6008051/pone_simulation/MC000004-nu_t...,2,4,4294967295,2,0,479.659520,1.745862,4.613300,0.143346,0.433998,-12,-2000001006,-12,-965.476213,-229.748926,-286.015491,4.343534e+05,992.435836,964.663916
2,/project/6008051/pone_simulation/MC000004-nu_t...,3,4,4294967295,3,0,231.925835,1.547583,3.101871,0.069249,0.554309,-12,-2000001006,-12,-2073.264946,-269.790218,-319.368051,3.353363e+05,2090.744915,508.542870
3,/project/6008051/pone_simulation/MC000004-nu_t...,4,4,4294967295,4,0,261.478809,2.144377,0.815938,0.325833,0.808219,-12,-2000001006,-12,233.049061,934.708189,381.993973,3.482837e+05,963.323031,908.411356
4,/project/6008051/pone_simulation/MC000004-nu_t...,5,4,4294967295,5,0,6118.632066,2.432875,1.925977,0.104540,0.342195,-12,-2000001006,-12,-92.615147,2598.574414,-1768.405608,1.227872e+06,2600.224327,1090.961638
5,/project/6008051/pone_simulation/MC000004-nu_t...,6,4,4294967295,6,0,2030.443942,1.964984,4.533721,0.201131,0.278803,-12,-2000001006,-12,85.543326,-3597.219941,-790.770521,8.049697e+05,3598.236924,955.132589
6,/project/6008051/pone_simulation/MC000004-nu_t...,7,4,4294967295,7,0,847.449454,2.670903,4.962259,0.139760,0.991729,-12,-2000001006,-12,-402.641290,-944.789840,-1599.081287,6.298772e+05,1027.009177,623.765666
7,/project/6008051/pone_simulation/MC000004-nu_t...,8,4,4294967295,8,0,157.005300,1.460940,4.936198,0.078207,0.195359,-12,-2000001006,-12,374.312149,1006.606314,63.963444,3.008502e+05,1073.948722,610.302760
8,/project/6008051/pone_simulation/MC000004-nu_t...,9,4,4294967295,9,0,523.460721,1.546919,4.056005,0.545401,0.227779,-12,-2000001006,-12,-800.016103,375.693043,685.321215,4.499105e+05,883.838801,1099.104222
9,/project/6008051/pone_simulation/MC000004-nu_t...,10,4,4294967295,10,0,2178.693019,1.331050,0.480010,0.117304,0.218911,-12,-2000001006,-12,3821.370480,2211.549368,189.130877,1.192667e+06,4415.180965,885.822207


In [5]:
print(f"total DAQ/EventProperties rows = {len(raw_event_properties_df)}")
print(f"unique RunID count = {raw_event_properties_df['RunID'].nunique()}")
print(f"unique EventID count = {raw_event_properties_df[['RunID', 'EventID', 'SubEventID']].drop_duplicates().shape[0]}")

for column in ["finalType1", "finalType2", "initialType"]:
    values = raw_event_properties_df[column].drop_duplicates().tolist()
    print(f"\nunique {column} ({len(values)}):")
    print(values)


total DAQ/EventProperties rows = 400
unique RunID count = 2
unique EventID count = 400

unique finalType1 (2):
[-12, 12]

unique finalType2 (1):
[-2000001006]

unique initialType (2):
[-12, 12]


to move:

mv /home/kbas/scratch/String340MC/Full_Geometry/Tau_PMT_Response/tau_gen_002.i3.gz /home/kbas/scratch/String340MC/Full_Geometry/Tau_PMT_Response/tau_gen_008.i3.gz /home/kbas/scratch/String340MC/Full_Geometry/Tau_PMT_Response/tau_gen_009.i3.gz /home/kbas/scratch/String340MC/Full_Geometry/Tau_PMT_Response/tau_gen_011.i3.gz /home/kbas/scratch/String340MC/Full_Geometry/Tau_PMT_Response/tau_gen_012.i3.gz /home/kbas/scratch/String340MC/Full_Geometry/Tau_PMT_Response/tau_gen_208.i3.gz /home/kbas/scratch/String340MC/Full_Geometry/Tau_PMT_Response/tau_gen_224.i3.gz /home/kbas/scratch/String340MC/Full_Geometry/Tau_PMT_Response/tau_gen_326.i3.gz /home/kbas/scratch/String340MC/Full_Geometry/Tau_PMT_Response/tau_gen_335.i3.gz /home/kbas/scratch/String340MC/Full_Geometry/Tau_PMT_Response/tau_gen_707.i3.gz /home/kbas/scratch/String340MC/Full_Geometry/Tau_PMT_Response/tau_gen_929.i3.gz /home/kbas/scratch/String340MC/Full_Geometry/Electron_PMT_Response/

to move

mv /home/kbas/scratch/String340MC/102_string/Tau_PMT_Response/tau_tau_gen_002.i3.gz /home/kbas/scratch/String340MC/102_string/Tau_PMT_Response/tau_tau_gen_008.i3.gz /home/kbas/scratch/String340MC/102_string/Tau_PMT_Response/tau_tau_gen_009.i3.gz /home/kbas/scratch/String340MC/102_string/Tau_PMT_Response/tau_tau_gen_011.i3.gz /home/kbas/scratch/String340MC/102_string/Tau_PMT_Response/tau_tau_gen_012.i3.gz /home/kbas/scratch/String340MC/102_string/Tau_PMT_Response/tau_tau_gen_208.i3.gz /home/kbas/scratch/String340MC/102_string/Tau_PMT_Response/tau_tau_gen_224.i3.gz /home/kbas/scratch/String340MC/102_string/Tau_PMT_Response/tau_tau_gen_326.i3.gz /home/kbas/scratch/String340MC/102_string/Tau_PMT_Response/tau_tau_gen_335.i3.gz /home/kbas/scratch/String340MC/102_string/Tau_PMT_Response/tau_tau_gen_707.i3.gz /home/kbas/scratch/String340MC/102_string/Tau_PMT_Response/tau_tau_gen_929.i3.gz /home/kbas/scratch/String340MC/102_string/Electron_PMT_Response/